In [129]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss

In [130]:
'''
Function to merge two dataframes
param df1: first dataframe
param df2: second dataframe
return: a single dataframe of the merged dataframes 
'''
def mergeDataframes(df1, df2):
    df = pd.concat([df1, df2])
    return df


'''
Function to merge tournament data for mens and womens and crop the data to match reg detail
params df1: dataframe of mens tournament data
params df2: dataframe of womens tournament data
return: dataframe of combined and cropped tournament data
'''
def mergeTournamentData(df1, df2):
    df1 = df1[df1['Season'] >= 2003].reset_index(drop=True)
    df2 = df2[df2['Season'] >= 2010].reset_index(drop=True)
    
    df = pd.concat([df1, df2])
    return df


'''
Function to split regular season detailed results into dataframes focused on outcome for one team
param df: regular season data
return: a dataframe where each team from a single row in reg data has its own row
'''
def regularDetailsFocus(df):
    RegWinners = pd.DataFrame()
    RegLossers = pd.DataFrame()

    # Establish new columns for that includes stats for one team
    columns = ['Season', 'TeamID', 'DayNum', 'Score', 'OppScore',
               'NumOT', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
               'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'OppFGM', 'OppFGA',
               'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppOR', 'OppDR', 'OppAst', 'OppTO',
               'OppStl', 'OppBlk', 'OppPF']

    # Split winners from regular season
    RegWinners[columns] = df[['Season', 'WTeamID', 'DayNum', 'WScore', 'LScore',
                              'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                              'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA',
                              'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO',
                              'LStl', 'LBlk', 'LPF']]

    # Add wins and losses columns
    RegWinners['Win'] = 1
    RegWinners['Loss'] = 0

    # Split lossers from regular season
    RegLossers[columns] = df[['Season', 'LTeamID', 'DayNum', 'LScore', 'WScore',
                               'NumOT', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                               'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WFGM', 'WFGA',
                               'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO',
                               'WStl', 'WBlk', 'WPF']]

    # Add wins and losses columns
    RegLossers['Win'] = 0
    RegLossers['Loss'] = 1

    # Combine all games into one dataframe
    AllRegDetail = pd.concat([RegWinners, RegLossers])
    
    return AllRegDetail


'''
Function to clean seed column
param seeds: Dataframe of historical seeds with column 'Seed'
return: A dataframe with the 'Seed' column converted to int
'''

def cleanSeed(seeds):
    seeds['Seed'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
    return seeds


'''
Function to join two Dataframes on 'TeamID'
param seeds: Dataframe of historical seeds with column 'Seed'
param tourny: Dataframe of compact tournament data with columns 'WTeamID' and 'LTeamID'
return: a single dataframe of the joined Dataframes
'''

def joinSeeds(seeds, tourny):
    seeds = seeds.set_index(['Season','TeamID'])

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'WSeed'}),
        on=['Season','WTeamID']
    )

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'LSeed'}),
        on=['Season','LTeamID']
    )
    return tourny


'''
Function to create the features from the regular season data at a single-game level
param df: dataframe of regular season data
return: dataframe of input features at a single-game level
'''
def createSingleFeatures(df):
    RegSeasonFeatures = pd.DataFrame()
    RegSeasonFeatures[['Season', 'TeamID', 'DayNum']] = df[['Season', 'TeamID', 'DayNum']]

    RegSeasonFeatures['PointRatio'] = df['Score'] / df['OppScore']  # Points ratio
    RegSeasonFeatures['MOV'] = df['Score'] - df['OppScore']  # Margin of victory
    RegSeasonFeatures['TORatio'] = df['TO'] / df['OppTO']  # Turnover ratio
    RegSeasonFeatures['FGM%'] = df['FGM'] / df['FGA']  # Scoring efficiency
    RegSeasonFeatures['FG3%M'] = df['FGM3'] / df['FGA3']  # 3-Point efficiency
    RegSeasonFeatures['FGA3%'] = df['FGA3'] / df['FGA']  # 3-Point attempt rate
    RegSeasonFeatures['FTM%'] = df['FTM'] / df['FTA']  # Free throw makes %
    RegSeasonFeatures['OppFTM%'] = df['OppFTM'] / df['OppFTA']  # Opponent free throw makes %
    RegSeasonFeatures['FTR'] = df['FTA'] / df['FGA']  # Free throw attempt rate
    RegSeasonFeatures['OppFTR'] = df['OppFTA'] / df['OppFGA']  # Opponent free throw attempt rate
    RegSeasonFeatures['ORRatio'] = df['OR'] / (df['OR'] + df['OppDR'])  # Offensive rebound ratio
    RegSeasonFeatures['DRRatio'] = df['DR'] / (df['DR'] + df['OppOR'])  # Defensive rebound ratio
    
    # New for 2026 (More advanced!)
    RegSeasonFeatures['NumPos'] = df['FGA'] - df['OR'] + df['TO'] + (0.44 * df['FTA'])  # ROUGH estimation of positions
    RegSeasonFeatures['OffEff'] = df['Score'] / RegSeasonFeatures['NumPos']  # Offensive efficiency
    RegSeasonFeatures['DefEff'] = df['OppScore'] / RegSeasonFeatures['NumPos']  # Defensive efficiency
    RegSeasonFeatures['NetEff'] = RegSeasonFeatures['OffEff'] - RegSeasonFeatures['DefEff']  # Net efficiency
    RegSeasonFeatures['TO%'] = df['TO'] / RegSeasonFeatures['NumPos']  # Turnover %
    RegSeasonFeatures['Ast%'] = df['Ast'] / df['FGM']  # Assist percentage
    RegSeasonFeatures['AstTORatio'] = df['Ast'] / df['TO']  # Assist to turnover ratio
    RegSeasonFeatures['AstRatio'] = df['Ast'] / df['OppAst']  # Assist ratio
    RegSeasonFeatures['OR%'] = df['OR'] / (df['FGA'] - df['FGM'])  # Offensive rebound %
    RegSeasonFeatures['DR%'] = df['DR'] / (df['OppFGA'] - df['OppFGM'])  # Defensive rebound %
    RegSeasonFeatures['EffFG%'] = (df['FGM'] + (0.5 * df['FGM3'])) / df['FGA']  # Effective field goal %
    RegSeasonFeatures['TS%'] = df['Score'] / (2 * (df['FGA'] + (0.44 * df['FTA'])))  # True shot %
    
    return RegSeasonFeatures

'''
Function to create the final features for input to the model
param df1: dataframe of single-game features
param df2: dataframe of regular season boxscore with columns ['Win', 'Loss']
return: dataframe of final input features
'''
def finalFeatures(df1, df2):
    # Season Average Features
    seasonAvg = df1.drop(columns='DayNum').groupby(['Season', 'TeamID']).mean().reset_index()
    
    # Get a win %
    winLoss = df2.groupby(['Season','TeamID'])[['Win', 'Loss']].sum().reset_index()
    winLoss['W/L'] = winLoss['Win'] / (winLoss['Win'] + winLoss['Loss'])  # Win/Loss ratio
    winLoss = winLoss.drop(['Win', 'Loss'], axis=1)
    
    # Merge season avg with win/loss
    finalSeasonAvg = pd.merge(seasonAvg, winLoss)
    
    # Rename columns appropriately
    finalSeasonAvg.columns = ['Season', 'TeamID'] + [f'Season_Avg_{col}' for col in finalSeasonAvg.columns if col not in ['Season', 'TeamID']]
    
    # Get the last 5 of the features
    lastFiveAvg = df1.sort_values(['Season', 'TeamID', 'DayNum']).groupby(['Season', 'TeamID']).tail(5).drop(columns='DayNum').groupby(['Season', 'TeamID']).mean().reset_index()
    lastFiveAvg.columns = ['Season', 'TeamID'] + [f'Last_5_Avg_{col}' for col in lastFiveAvg.columns if col not in ['Season', 'TeamID']]

    # Merge the season with rolling
    features = pd.merge(finalSeasonAvg, lastFiveAvg)
    return features


'''
Function to split tournament data to match the format of input features
param df: dataframe of compact tournament data
return: dataframe of tournament data ready to merge with features
'''
def splitHistTournamentData(df):
    WTourney = pd.DataFrame()
    LTourney = pd.DataFrame()
    
    # Setup separate dataframes for tournament data and add target feature
    WTourney[['Season', 'Team1', 'Team2', 'Team1Seed', 'Team2Seed']] = df[['Season', 'WTeamID', 'LTeamID', 'WSeed', 'LSeed']]
    WTourney['Result'] = 1

    LTourney[['Season', 'Team1', 'Team2', 'Team1Seed', 'Team2Seed']] = df[['Season', 'LTeamID', 'WTeamID', 'LSeed', 'WSeed']]
    LTourney['Result'] = 0

    # Join individual together
    TourneyInput = pd.concat([WTourney, LTourney])
    return TourneyInput


'''
Function to calculate difference in teams input features for historical tournament matchups for LR
param df1: dataframe of tournament data
param df2: dataframe of input features
return: dataframe of differences for training
'''
def calculateDifferenceHistTourn(df1, df2):
    # Merge two team stats from RegSeasonFeatures
    TourneyFinal = df1.merge(df2, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], suffixes=('', '_T1'))
    TourneyFinal = TourneyFinal.merge(df2, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], suffixes=('_T1', '_T2'))
    TourneyFinal[['Seed_T1', 'Seed_T2']] = TourneyFinal[['Team1Seed', 'Team2Seed']]

    # Calculate the differences (Team1 - Team2) for the features for input to logistic regression
    featureCols = [col for col in df2 if col not in ['Season', 'TeamID']] + ['Seed']
    for col in featureCols:
        TourneyFinal[col + '_Diff'] = TourneyFinal[col + '_T1'] - TourneyFinal[col + '_T2']

    # Drop all _T1 and _T2, keep only _Diff and Result
    TourneyFinal = TourneyFinal[['Season', 'Team1', 'Team2'] + [col + '_Diff' for col in featureCols] + ['Result']]
    return TourneyFinal

In [131]:
# Mens data import
mRegDetail = pd.read_csv('data/men/MRegularSeasonDetailedResults.csv')
mTournCompact = pd.read_csv('data/men/MNCAATourneyCompactResults.csv')
mTournSeeds = pd.read_csv('data/men/MNCAATourneySeeds.csv')
mNames = pd.read_csv('data/men/MTeamSpellings.csv')

# Womens data import
wRegDetail = pd.read_csv('data/women/WRegularSeasonDetailedResults.csv')
wTournCompact = pd.read_csv('data/women/WNCAATourneyCompactResults.csv')
wTournSeeds = pd.read_csv('data/women/WNCAATourneySeeds.csv')
wNames = pd.read_csv('data/women/WTeamSpellings.csv') 

# Clean and merge seeds with tournament results
mCleanSeeds = cleanSeed(mTournSeeds.copy())
wCleanSeeds = cleanSeed(wTournSeeds.copy())
mFullTourn = joinSeeds(mCleanSeeds, mTournCompact)
wFullTourn = joinSeeds(wCleanSeeds, wTournCompact)

# Combined data
regDetail = mergeDataframes(mRegDetail, wRegDetail)
compactTourn = mergeTournamentData(mFullTourn, wFullTourn)
names = mergeDataframes(mNames, wNames)

# Split regular season detailed results into dataframes focused on outcome for one team
allRegDetail = regularDetailsFocus(regDetail)

# Create single-game features
singleFeatures = createSingleFeatures(allRegDetail)

# Create season avg and 5 game window avg features
finalAvgFeatures = finalFeatures(singleFeatures, allRegDetail)

# Get compact tournament setup like features
tourneyInput = splitHistTournamentData(compactTourn)

# Prepare input data with target
tourneyFinal = calculateDifferenceHistTourn(tourneyInput, finalAvgFeatures)